# Skill documents, evolved

Can an outer run improve *what an agent is told* -- a set of `skills/<name>.md`
documents -- while the budget, the episode list, and the scoring all stay
outside what a candidate can touch?

Every cell below imports the committed example's own modules and drives its own
functions through the deterministic fixture agent the script injects, so
nothing here needs a credential, a network, or a dollar. The [example
README](https://github.com/sentient-xyz/meta-evolve/blob/main/examples/research/context_evolution/README.md) is the canonical explanation; this is a place to poke at
it.

In [1]:
import sys
from pathlib import Path

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
example = root / "examples/research/context_evolution"
if str(example) not in sys.path:
    sys.path.insert(0, str(example))

from inner_agent import InnerAgent
from skill_context import FIXTURE_AGENT, SKILL_BUDGET, SKILL_CALL_CEILING
from skill_context import evolve_skills
from skill_episodes import EPISODES

print("episodes:", [episode.name for episode in EPISODES])
print("the agent declares itself:", FIXTURE_AGENT.model)

episodes: ['interest', 'leases']
the agent declares itself: deterministic-sdk-fixture-v1


## Run the search

Three evaluations: the empty seed, then two proposals that each add one whole
skill document. There is one skill per episode, so a candidate holding neither
scores `0.0`, one scores `0.5`, and both score `1.0`.

The wrapper below *remembers*, which is what a live SDK client is, and records
every task each session was shown. Both numbers matter and they say different
things. Six sessions is the arithmetic: one per episode, not one per candidate,
which is why the declared ceiling counts proposals. **One task per session** is
the property: both notes are tagged by a policy that appears in no
workspace, so a session spanning them would let the second answer from the first --
and that reads exactly like the skill set carrying the knowledge.

`InnerAgent` is how the wrapper arrives: the factory bound to what it declares
itself to be. A model id alone would not do, and the next section is why.

In [2]:
sessions = []


def remembering_agent(instructions):
    """A stateful wrapper: if a session spans two episodes, it will show."""

    shown = []
    sessions.append(shown)
    inner = FIXTURE_AGENT.build(instructions)

    def propose(tree, *, context):
        shown.append(next(path for path in sorted(tree) if path != "README.md"))
        return inner(tree, context=context)

    return propose


stateful = InnerAgent(build=remembering_agent, model=FIXTURE_AGENT.model,
                      revision="wrapper-v1")
run = evolve_skills(stateful)
print("skills:", list(run.best().value))
print("solved:", run.best_trial().metrics["solved"])
print("evaluations:", run.summary().evaluation_count)
print("sessions:", len(sessions), "== declared ceiling:", SKILL_CALL_CEILING)
print("tasks any one session saw:", sorted({len(shown) for shown in sessions}))
print("declared budget:", SKILL_BUDGET)

skills: ['skills/interest.md', 'skills/leases.md']
solved: 1.0
evaluations: 3
sessions: 6 == declared ceiling: 6
tasks any one session saw: [1]
declared budget: Budget(evaluations=3, trials=2, tokens=None, spend_micros=None, wall_seconds=None)


## The losers are still there

`run.trials()` keeps every candidate in order, so the climb is inspectable
rather than summarised. The intermediate `0.5` is what makes the run evidence
about the *skill text*: the fixture agent never changes, so nothing else could
have moved the score.

In [3]:
[
    (sorted(trial.artifact.value), trial.metrics["solved"])
    for trial in run.trials()
]

[([], 0.0),
 (['skills/leases.md'], 0.5),
 (['skills/interest.md', 'skills/leases.md'], 1.0)]

## Why a model id is not an agent

The skill set is the only thing a proposal changes. The budget, the episode
list, and the evaluator live in the declaration above it -- and the evaluator
declares what it was configured with, because the agent lives inside its
closure where a run-local reference cannot see it.

Below, two agents declare **the same model, the same revision, and the same
configuration**, and differ only in the wrapper behind them. That is the point
of the cell: every string a caller can claim is held fixed, so the separation
has to come from the factory itself. Wrapper revision, SDK version, tool
permissions and turn limits are all things a model id cannot see -- but a
caller who declares them identically for two different implementations still
gets two versions, because the factory's own code is hashed too.

The cell prints both halves: that the declared fields are equal, and that the
verified `implementation` digest is not.

In [4]:
import meta_evolve as meta


def indifferent(instructions):
    """Same declaration, different wrapper: this one ignores what it was told."""

    def propose(tree, *, context):
        return tree

    return propose


twin = InnerAgent(build=indifferent, model=stateful.model,
                  revision=stateful.revision,
                  configuration=stateful.configuration)
print("declared alike:", all(getattr(twin, field) == getattr(stateful, field)
                             for field in ("model", "revision", "configuration")))
print("same implementation:",
      twin.manifest["implementation"] == stateful.manifest["implementation"])
other = evolve_skills(twin)
print("one model id:", FIXTURE_AGENT.model)
print("two outcomes:", run.summary().primary_score, "and",
      other.summary().primary_score)
print("same run id:", run.id == other.id)
try:
    run.compare(other)
except meta.RunsNotComparable as refusal:
    print("refused on:", refusal.declaration)

declared alike: True
same implementation: False
one model id: deterministic-sdk-fixture-v1
two outcomes: 1.0 and 0.0
same run id: False
refused on: evaluator component


## A failure is not a low score

If the wrapper cannot produce a candidate at all, there is nothing to rank. The
suite reports a typed failure carrying **no metric**, so it can never out-rank a
skill set that merely scored badly -- and the fault each episode recorded is
named rather than flattened into a zero.

In [5]:
from skill_suite import evaluate_skills


def silent_agent(_instructions):
    """A wrapper that cannot answer. Raising is the absence of a candidate."""

    def propose(_tree, *, context):
        raise RuntimeError("wrapper produced nothing")

    return propose


outcome = evaluate_skills(
    meta.SkillSet(),
    InnerAgent(build=silent_agent, model="silent-v1", revision="silent-v1"),
)
print("metrics:", dict(outcome.metrics))
print("unfinished:", dict(outcome.failure.details["unfinished_episodes"]))

metrics: {}
unfinished: {'interest': FrozenMap({'detail': 'proposer_failure', 'kind': 'wrapper-failed', 'reported': 'proposer_failure'}), 'leases': FrozenMap({'detail': 'proposer_failure', 'kind': 'wrapper-failed', 'reported': 'proposer_failure'})}


## Change and predict

This runs in a cell. No repository file to edit, no kernel to restart -- the
wrapper is yours, and `InnerAgent` is how you hand it over.

Write one that hears the interest policy and is deaf to the lease
one, then predict all three trial scores *before* running it:

```python
def half_deaf(instructions):
    inner = FIXTURE_AGENT.build(instructions)

    def propose(tree, *, context):
        return inner(tree, context=context) if "note_05_interest.tag" in tree else tree

    return propose


half = evolve_skills(
    InnerAgent(build=half_deaf, model="half-deaf-v1", revision="half-deaf-v1")
)
[trial.metrics["solved"] for trial in half.trials()]
```

Then the harder question: when the score stops moving, did the search get
worse, or did the two episodes stop disagreeing about what a skill has to say?

The ACE sibling is in [`ace.ipynb`](https://github.com/sentient-xyz/meta-evolve/blob/main/docs/notebooks/context_evolution/ace.ipynb) -- same boundary, a
representation that can cite and retire one entry at a time.